In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")                # It builds the full file path to q1.csv  by safely joining
                                                            # the base directory stored in path with the filename,
                                                            # so the code works correctly across operating systems.
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()


In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns=['Order_ID'])
df.head()

In [ ]:
# Task 2: Write your code here:
# 2. Do we have missing values?
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

# we have missing data, hence we need to handle them!
cols = ['Delivery_Time', 'Preparation_Time_min', 'Vehicle_Type', 'Time_of_Day', 'Traffic_Level', 'Weather', 'Distance_km', 'Courier_Experience_yrs']
df_clean = df[cols].copy()

# Drop rows where target (price) or key features are missing - can't predict without them
print(f"Before: {df_clean.shape}")
df_clean = df_clean.dropna(subset=['Delivery_Time', 'Preparation_Time_min', 'Vehicle_Type'])
print(f"After dropping missing Delivery_Time/Preparation_Time_min/Vehicle_Type: {df_clean.shape}")

# Fill categorical columns with 'unknown' - missing likely means "not specified"
for x in ['Weather', 'Traffic_Level', 'Time_of_Day', 'Distance_km']:
    df_clean[x] = df_clean[x].fillna('unknown')

# Fill cylinders with mode - discrete feature, mode is most representative
for x in ['Preparation_Time_min', 'Courier_Experience_yrs', 'Time_of_Day', 'Vehicle_Type']:
    df_clean[x] = df_clean[x].fillna(df_clean[x].mode()[0])


print("Missing values remaining:", df_clean.isnull().sum().sum())


In [ ]:
# Task 3: Write your code here:
# 3. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder
# Encode categorical columns - converts text to integers
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler
features = df_clean.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[features] = scaler.fit_transform(df_clean[features])
df_clean.head()

In [ ]:
# Task 6: Write your code here:
#no need this is a regression problem we just have to check if data is skewed which we already did

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error as sklearn_mae
from sklearn.ensemble import RandomForestRegressor
def gradient_descent(X, y, learning_rate, n_iters=500):
  m, n = X.shape  # m rows, n columns (dimensions)
  theta = np.zeros(n)  # initialize a zeros weight vector with n dimensions
  losses = []

  for _ in tqdm(range(n_iters), desc="Training Linear Regression"):
    y_hat = np.dot(X, theta)
    gradient = np.dot(X.T, (y_hat - y)) / m
    theta -= learning_rate * gradient

    loss = sklearn_mae(y, y_hat)
    losses.append(loss)

  return theta, losses




model = RandomForestRegressor(n_estimators=200)



modelloss= 0.0

n_splits = 5 # K=5 Folds
# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
lr_losses = []
lr_mae = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(X,y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")
  print("training RandomForestRegressor")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]


  # Train
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)

  # Calculate metrics
  mae = sklearn_mae(y_test, y_pred)


    # Store results
  modelloss=+ mae

print(modelloss)



In [ ]:
# Task 1: Write your code here:


# Feature importance
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:


print("Plot predicted delivery time histogram:")
cm = plt.hist((y_test, y_pred))
plt.figure(figsize=(6, 5))

plt.show()




In [ ]:
# Task Bonus: Write your code here: